# 11.05 -- Edge and On-Device Deployment: llama.cpp, ONNX, and MLC

**Module:** 11 -- LLM Systems Engineering  
**Notebook:** 5 of 5  
**Prerequisites:** 09.04 (Quantisation), 11.01 (GPU Memory Planning)

---

## Learning Objectives

1. Understand the unique constraints of edge deployment: memory, power, latency, and offline operation
2. Understand the GGUF format and how llama.cpp achieves efficient CPU inference
3. Implement a device capability profiler that determines which models can run on target hardware
4. Understand ONNX export and runtime for cross-platform model deployment
5. Compare edge deployment options across frameworks, hardware, and quality tiers

---

## 1. Edge Deployment Constraints

Server-side LLM inference (Module 09, 10, 11.01-11.04) optimises for throughput,
latency percentiles, and GPU utilisation. Edge deployment has an entirely different
constraint profile:

```
  Server (cloud)               Edge (device)
  -------------------          -------------------
  HBM bandwidth: 2 TB/s        LPDDR5: 68 GB/s  (30x slower)
  GPU memory:    40-80 GB      RAM: 8-16 GB      (5-10x less)
  Power:         300-700 W     Battery: 15-45 W  (20x less)
  Connectivity:  always-on     Offline: must work
  Cost:          per-token $   One-time device $
  Latency SLO:   P99 < 1s      Perceived smooth: > 5 tok/s
```

The combination of slower memory and smaller capacity makes quantisation not
just useful on the edge -- it is mandatory. A 7B model in FP16 requires 14 GB,
which exceeds most mobile RAM budgets. The same model in Q4_K_M (4-bit) fits in 4 GB.

---

## 2. The GGUF Format and llama.cpp

**GGUF** (GPT-Generated Unified Format) is the file format used by llama.cpp.
It stores model weights in a single binary file with:
- A header containing all model metadata (architecture, tokeniser, hyperparameters)
- Weight tensors in one of 20+ quantisation formats (Q2_K, Q4_K_M, Q6_K, etc.)
- Support for mixed-precision: attention layers in higher bit-width than MLP layers

**llama.cpp** is a C/C++ inference engine designed for CPU execution with optional
GPU offloading. Key design decisions:
- Pure C: no Python runtime, no CUDA dependency (ships on any platform)
- SIMD-optimised kernels: uses AVX2/NEON for matrix multiplication
- KV cache with quantisation: INT8 KV cache halves memory vs FP16
- GPU offloading: send N transformer layers to GPU, rest on CPU

```
GGUF quantisation tiers for a 7B model:
  Q2_K:   2.8 GB   fast, significant quality loss
  Q4_K_S: 4.1 GB   balanced -- common choice for 8 GB devices
  Q4_K_M: 4.4 GB   slightly better quality, recommended default
  Q6_K:   5.9 GB   near-lossless, good for 8+ GB devices
  Q8_0:   7.7 GB   essentially lossless, needs 8+ GB RAM
  F16:   14.0 GB   full precision -- too large for most devices
```


In [ ]:
# Environment setup.
#
# Edge deployment analysis is primarily a system design and measurement discipline.
# We implement:
#   1. A device capability profiler (CPU, RAM, theoretical bandwidth)
#   2. A model sizing calculator for GGUF formats
#   3. An ONNX export pipeline for GPT-2 as a concrete example
#   4. A cross-framework benchmarking harness

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import platform
import psutil
import os
import time
import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

print('System profile:')
print(f'  Platform:    {platform.system()} {platform.machine()}')
print(f'  CPU:         {platform.processor() or "(unknown)"}')
print(f'  CPU cores:   {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical')
mem = psutil.virtual_memory()
print(f'  RAM total:   {mem.total / 1e9:.1f} GB')
print(f'  RAM free:    {mem.available / 1e9:.1f} GB')
print(f'  GPU:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')


## 3. Device Capability Profiler


In [ ]:
# Device capability profiler.
#
# Before deploying to an edge device, we need to determine:
#   1. How much RAM is available for the model?
#   2. What is the memory bandwidth? (determines tok/s)
#   3. Does the device have a GPU/NPU that can be used?
#   4. What quantisation level is needed to fit the target model?
#
# Memory bandwidth is the dominant determinant of decode speed on edge devices.
# The decode roofline (from 10.05) gives an upper bound:
#   max_tokens_per_second = bandwidth_GB_s / model_size_GB
# e.g. MacBook M3 (100 GB/s, 7B Q4_K_M = 4.4 GB): ~22 tok/s
#      iPhone 15 Pro (68 GB/s, 7B Q4_K_M = 4.4 GB): ~15 tok/s
#      Raspberry Pi 5 (17 GB/s, 7B Q4_K_M = 4.4 GB): ~3.9 tok/s (barely usable)

@dataclass
class DeviceSpec:
    name:           str
    ram_gb:         float
    bandwidth_gb_s: float    # memory bandwidth (HBM or LPDDR)
    cpu_gflops:     float    # FP32 peak compute
    has_npu:        bool = False
    npu_tops:       float = 0.0   # neural processing unit throughput
    os:             str = 'linux'


DEVICE_SPECS = [
    DeviceSpec('MacBook M3 Pro',        36,  150,  3_800,  True,  38,   'macos'),
    DeviceSpec('MacBook M2 (8 GB)',      8,  100,  2_600,  True,  15,   'macos'),
    DeviceSpec('iPhone 15 Pro',          8,   68,  4_350,  True,  35,   'ios'),
    DeviceSpec('Pixel 9 Pro',            12,  68,  2_100,  True,  45,   'android'),
    DeviceSpec('Raspberry Pi 5',         8,   17,    192,  False, 0.0,  'linux'),
    DeviceSpec('Jetson Orin Nano',       8,   68,  1_024,  True,  40,   'linux'),
    DeviceSpec('Jetson AGX Orin',       64,  204,  5_632,  True, 275,   'linux'),
    DeviceSpec('AMD Ryzen 9 7950X',     64,   83,  4_200,  False, 0.0,  'linux'),
    DeviceSpec('Intel i9-13900K',       64,   90,  4_000,  False, 0.0,  'linux'),
]


# GGUF quantisation presets: (bits, quality loss pct, size multiplier vs FP16)
GGUF_QUANTS = {
    'Q2_K':   (2.0,  8.0, 0.200),
    'Q4_K_S': (4.0,  2.5, 0.293),
    'Q4_K_M': (4.5,  1.5, 0.315),
    'Q5_K_M': (5.5,  0.8, 0.385),
    'Q6_K':   (6.0,  0.3, 0.421),
    'Q8_0':   (8.0,  0.1, 0.551),
    'F16':    (16.0, 0.0, 1.000),
}


def model_gguf_size_gb(n_params: int, quant: str) -> float:
    """Estimate GGUF file size for a model at a given quantisation level."""
    _, _, multiplier = GGUF_QUANTS[quant]
    fp16_size_gb = n_params * 2 / 1e9
    return fp16_size_gb * multiplier


def max_tokens_per_second(device: DeviceSpec, model_gb: float) -> float:
    """
    Estimate decode throughput using the memory bandwidth roofline.

    At each decode step, the entire model is read from memory once.
    tokens_per_second = bandwidth / model_size
    NPU accelerates this by 2-4x if available (rough estimate).
    """
    base_tps = device.bandwidth_gb_s / model_gb
    if device.has_npu and device.npu_tops > 0:
        # NPU provides partial acceleration; rough 1.5-2x multiplier
        base_tps *= 1.7
    return base_tps


def recommend_quant(device: DeviceSpec, n_params: int,
                    min_tps: float = 5.0) -> Optional[str]:
    """
    Find the highest-quality quantisation that:
      (a) fits in device RAM (with 20% headroom for OS and runtime)
      (b) achieves at least min_tps tokens/second
    Returns None if no quantisation level is feasible.
    """
    available_gb = device.ram_gb * 0.75   # 25% headroom

    # Try quants from highest to lowest quality
    quant_order = ['F16', 'Q8_0', 'Q6_K', 'Q5_K_M', 'Q4_K_M', 'Q4_K_S', 'Q2_K']
    for quant in quant_order:
        size_gb = model_gguf_size_gb(n_params, quant)
        if size_gb > available_gb:
            continue
        tps = max_tokens_per_second(device, size_gb)
        if tps >= min_tps:
            return quant
    return None


# Print capability matrix for 7B and 1B models
for model_name, n_params in [('7B model', 7_000_000_000), ('1B model', 1_200_000_000)]:
    print(f'\n{model_name}  -- device feasibility:')
    print(f'  {"Device":<24} {"Rec. quant":<12} {"Size GB":>8} {"Est. tok/s":>10}')
    print('  ' + '-' * 60)
    for dev in DEVICE_SPECS:
        quant = recommend_quant(dev, n_params)
        if quant:
            size  = model_gguf_size_gb(n_params, quant)
            tps   = max_tokens_per_second(dev, size)
            print(f'  {dev.name:<24} {quant:<12} {size:>8.1f} {tps:>10.1f}')
        else:
            print(f'  {dev.name:<24} {"NOT FEASIBLE":<12}')


In [ ]:
# ONNX export: cross-platform model portability.
#
# ONNX (Open Neural Network Exchange) is an open format for representing
# ML models. Once exported to ONNX, a model can run on:
#   - CPUs via ONNX Runtime (cross-platform, optimised for x86 and ARM)
#   - GPUs via ONNX Runtime CUDA/TensorRT execution providers
#   - Mobile via CoreML (iOS) or NNAPI (Android) through conversion tools
#   - NPUs via Qualcomm QNN or MediaTek APU SDKs
#
# We export GPT-2 to ONNX and verify that the ONNX output matches PyTorch.
# For production LLM deployment, tools like Optimum (HuggingFace) handle the
# full export including KV cache inputs/outputs for autoregressive generation.

import subprocess
import sys

# Install onnx and onnxruntime if not present (lightweight install)
try:
    import onnx
    import onnxruntime as ort
    print('onnx and onnxruntime already installed')
except ImportError:
    print('Installing onnx and onnxruntime...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnx', 'onnxruntime', '-q'],
                   check=True)
    import onnx
    import onnxruntime as ort
    print('Installed successfully')

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model_pt = AutoModelForCausalLM.from_pretrained('gpt2')
model_pt.eval()

# Export a single forward pass (not the full autoregressive loop)
ONNX_PATH = '/tmp/gpt2_decoder.onnx'

prompt_ids  = tokenizer('The transformer', return_tensors='pt').input_ids
input_shape = prompt_ids.shape

print(f'Exporting GPT-2 to ONNX (input shape: {input_shape})...')
with torch.no_grad():
    torch.onnx.export(
        model_pt,
        (prompt_ids,),
        ONNX_PATH,
        input_names=['input_ids'],
        output_names=['logits'],
        dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'},
                      'logits':    {0: 'batch', 1: 'seq'}},
        opset_version=17,
        verbose=False,
    )

onnx_size_mb = os.path.getsize(ONNX_PATH) / 1024**2
print(f'ONNX model saved: {onnx_size_mb:.1f} MB')

# Validate ONNX model
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print('ONNX model validation: OK')

# Run inference with ONNX Runtime and compare to PyTorch
sess_options = ort.SessionOptions()
sess_options.log_severity_level = 3   # suppress INFO logs
sess = ort.InferenceSession(ONNX_PATH, sess_options)

onnx_input  = {'input_ids': prompt_ids.numpy()}
onnx_logits = sess.run(['logits'], onnx_input)[0]

with torch.no_grad():
    pt_logits = model_pt(prompt_ids).logits.numpy()

max_err = abs(onnx_logits - pt_logits).max()
print(f'\nONNX vs PyTorch logit comparison:')
print(f'  Max absolute difference: {max_err:.4f}')
print(f'  Verified (max_err < 0.01): {max_err < 0.01}')


In [ ]:
# Edge deployment benchmark and comparison dashboard.
#
# We measure real CPU inference throughput for GPT-2 in two modes:
#   1. PyTorch FP32 (standard)
#   2. ONNX Runtime (optimised for CPU)
# Then we compare projected edge performance for 1B and 7B models across devices.

def timed_inference(fn, n_runs=5, n_warmup=2):
    """Time a zero-argument callable over multiple runs."""
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.std(times)


# Benchmark: single forward pass latency
test_input    = tokenizer('The attention mechanism', return_tensors='pt').input_ids
test_input_np = test_input.numpy()

pt_mean, pt_std = timed_inference(
    lambda: model_pt(test_input), n_runs=10, n_warmup=3
)
ort_mean, ort_std = timed_inference(
    lambda: sess.run(['logits'], {'input_ids': test_input_np}), n_runs=10, n_warmup=3
)

print(f'Single forward pass latency (CPU, GPT-2, seq=4):')
print(f'  PyTorch FP32:   {pt_mean:.1f} +/- {pt_std:.1f} ms')
print(f'  ONNX Runtime:   {ort_mean:.1f} +/- {ort_std:.1f} ms')
print(f'  ORT speedup:    {pt_mean / max(ort_mean, 0.001):.2f}x')


fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.32)

# Panel 1: GGUF quantisation: size vs quality
ax = fig.add_subplot(gs[0, 0])
n_params_7b = 7_000_000_000
quant_names = list(GGUF_QUANTS.keys())
quant_sizes = [model_gguf_size_gb(n_params_7b, q) for q in quant_names]
quant_loss  = [GGUF_QUANTS[q][1] for q in quant_names]
sc = ax.scatter(quant_sizes, quant_loss, s=160, c=quant_sizes,
                cmap='RdYlGn_r', zorder=5, vmin=2, vmax=16)
plt.colorbar(sc, ax=ax, label='Model size (GB)')
for name, size, loss in zip(quant_names, quant_sizes, quant_loss):
    ax.annotate(name, (size, loss), xytext=(0.15, 0.08),
                textcoords='offset points', fontsize=9)
ax.axvline(8, color='gray', ls='--', alpha=0.5, label='8 GB device RAM (75% threshold)')
ax.set_xlabel('Model size in GGUF (GB)', fontsize=11)
ax.set_ylabel('Quality loss vs FP16 (%)', fontsize=11)
ax.set_title('7B Model: Quantisation Size vs Quality Trade-off', fontsize=12)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 2: estimated tok/s per device for 7B Q4_K_M
ax = fig.add_subplot(gs[0, 1])
size_7b_q4 = model_gguf_size_gb(7_000_000_000, 'Q4_K_M')
dev_names  = [d.name.replace(' ', '\n') for d in DEVICE_SPECS]
dev_tps    = []
dev_fits   = []
for dev in DEVICE_SPECS:
    avail = dev.ram_gb * 0.75
    if size_7b_q4 <= avail:
        dev_tps.append(max_tokens_per_second(dev, size_7b_q4))
        dev_fits.append(True)
    else:
        dev_tps.append(0)
        dev_fits.append(False)
colors_d = ['#2ca02c' if f else '#d62728' for f in dev_fits]
bars = ax.barh(dev_names, dev_tps, color=colors_d, alpha=0.85)
ax.axvline(5, color='gray', ls='--', lw=1.5, label='Minimum usable (5 tok/s)')
ax.axvline(15, color='#2ca02c', ls=':', lw=1.5, label='Comfortable (15 tok/s)')
for bar, tps, fits in zip(bars, dev_tps, dev_fits):
    if fits and tps > 0:
        ax.text(tps + 0.2, bar.get_y() + bar.get_height()/2,
                f'{tps:.0f} tok/s', va='center', fontsize=8)
    elif not fits:
        ax.text(0.5, bar.get_y() + bar.get_height()/2,
                'OOM', va='center', fontsize=8, color='red', fontweight='bold')
ax.set_xlabel('Estimated tok/s (bandwidth roofline)', fontsize=11)
ax.set_title('7B Q4_K_M Decode Speed by Device', fontsize=12)
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis='x')

# Panel 3: framework comparison: latency overhead
ax = fig.add_subplot(gs[1, 0])
frameworks  = ['PyTorch\nFP32', 'ONNX\nRuntime', 'llama.cpp\n(simulated)', 'TFLite\n(simulated)', 'CoreML\n(simulated)']
norm_factor = pt_mean
# Real measurements for PyTorch and ORT; simulated ratios for others
rel_latency = [1.0, ort_mean/norm_factor, 0.45, 0.60, 0.35]
bar_colors  = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
bars = ax.bar(frameworks, rel_latency, color=bar_colors, alpha=0.85)
for b, v in zip(bars, rel_latency):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.2f}x',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Relative latency (1.0 = PyTorch FP32)', fontsize=11)
ax.set_title('Inference Framework Latency Comparison\n(Measured: PyTorch, ORT; Simulated: others)', fontsize=11)
ax.grid(alpha=0.3, axis='y')

# Panel 4: memory vs accuracy Pareto for deployment tiers
ax = fig.add_subplot(gs[1, 1])
tiers = [
    ('Cloud FP16\n70B',      140,  99.0, 'server'),
    ('Cloud INT8\n13B',       13,  97.5, 'server'),
    ('Desktop FP16\n7B',      14,  96.0, 'desktop'),
    ('Desktop Q6\n7B',         6,  95.5, 'desktop'),
    ('Mobile Q4\n7B',          4,  93.5, 'mobile'),
    ('Mobile Q4\n3B',          2,  91.0, 'mobile'),
    ('Edge Q4\n1B',           0.7, 85.0, 'edge'),
    ('Embedded Q2\n1B',       0.4, 78.0, 'embedded'),
]
marker_map = {'server':'D', 'desktop':'s', 'mobile':'o', 'edge':'^', 'embedded':'v'}
color_map  = {'server':'#1f77b4','desktop':'#2ca02c','mobile':'#ff7f0e','edge':'#d62728','embedded':'#9467bd'}
for label, mem_gb, acc, tier in tiers:
    ax.scatter(mem_gb, acc, marker=marker_map[tier], color=color_map[tier],
               s=140, zorder=5)
    ax.annotate(label, (mem_gb, acc), xytext=(0.1, 0.2),
                textcoords='offset points', fontsize=7.5)
ax.set_xlabel('Model memory (GB)', fontsize=11)
ax.set_ylabel('Relative accuracy (% of GPT-4 on MMLU)', fontsize=11)
ax.set_title('Deployment Tier Pareto: Memory vs Accuracy', fontsize=12)
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker=marker_map[t], color=color_map[t],
                          ls='None', ms=8, label=t.title())
                   for t in ['server','desktop','mobile','edge','embedded']]
ax.legend(handles=legend_elements, fontsize=8)
ax.set_xscale('log')
ax.grid(alpha=0.3)

# Panel 5: llama.cpp GPU offload benefit
ax = fig.add_subplot(gs[2, 0])
# Simulate: model has 32 layers; each layer offloaded to GPU speeds up that layer ~10x
# Remaining layers run on CPU; total throughput is harmonic mean
layers_total = 32
cpu_layer_ms = 2.0    # ms per layer on CPU (bandwidth bound)
gpu_layer_ms = 0.15   # ms per layer on GPU (much faster HBM)

offload_layers = range(0, 33, 2)
est_tps = []
for n_gpu in offload_layers:
    n_cpu = layers_total - n_gpu
    total_ms = n_cpu * cpu_layer_ms + n_gpu * gpu_layer_ms
    est_tps.append(1000 / total_ms)

ax.plot(list(offload_layers), est_tps, 'o-', color='#2ca02c', lw=2.5, ms=6)
ax.axhline(est_tps[0], color='#d62728', ls='--', lw=1.5, label='All CPU baseline')
ax.axhline(est_tps[-1], color='#1f77b4', ls=':', lw=1.5, label='All GPU upper bound')
ax.set_xlabel('Layers offloaded to GPU (n_gpu_layers)', fontsize=11)
ax.set_ylabel('Estimated tok/s', fontsize=11)
ax.set_title('llama.cpp GPU Offload Benefit\n(7B model, 32 layers, MacBook M-series style)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 6: ORT vs PyTorch measured latency (real data)
ax = fig.add_subplot(gs[2, 1])
framework_labels = ['PyTorch FP32', 'ONNX Runtime']
means  = [pt_mean,  ort_mean]
stds   = [pt_std,   ort_std]
cols   = ['#d62728', '#2ca02c']
bars   = ax.bar(framework_labels, means, yerr=stds, color=cols, alpha=0.85,
                capsize=6, width=0.4)
for b, v in zip(bars, means):
    ax.text(b.get_x() + b.get_width()/2, v + max(stds) * 1.1,
            f'{v:.1f} ms', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Latency (ms) per forward pass', fontsize=11)
ax.set_title(f'Measured CPU Latency: GPT-2 (117M)\n'
             f'ORT speedup: {pt_mean/max(ort_mean,0.001):.1f}x', fontsize=12)
ax.grid(alpha=0.3, axis='y')

plt.suptitle('Edge and On-Device LLM Deployment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/11_edge_deployment.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Engineering Notes

**Choosing the right edge framework**

| Framework | Best for | Limitations |
|-----------|----------|------------|
| llama.cpp | CPU/mixed, Linux/macOS/Windows | No mobile app integration |
| ONNX Runtime | Cross-platform, easiest export | Larger overhead than llama.cpp |
| MLC-LLM | iOS/Android/WebGPU | Complex setup, fewer models |
| TFLite | Android (NNAPI), small models | Limited LLM architecture support |
| Core ML | Apple ecosystem (iPhone, Mac) | Apple-only |
| ExecuTorch | PyTorch-native mobile | Early stage, limited tooling |

**Practical llama.cpp deployment checklist**

```bash
# 1. Download GGUF model
huggingface-cli download bartowski/Llama-3.2-1B-Instruct-GGUF \
    Llama-3.2-1B-Instruct-Q4_K_M.gguf

# 2. Build llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && cmake -B build -DLLAMA_METAL=ON && cmake --build build -j

# 3. Run with 20 GPU layers offloaded (MacBook M-series)
./build/bin/llama-cli \
    -m Llama-3.2-1B-Instruct-Q4_K_M.gguf \
    -n 256 -ngl 20 \
    -p 'Explain attention mechanisms in one paragraph:'

# 4. Start an OpenAI-compatible server
./build/bin/llama-server \
    -m Llama-3.2-1B-Instruct-Q4_K_M.gguf \
    --port 8080 -ngl 20 -c 4096
```

## 5. Module 11 Summary

This module covered the production systems engineering layer:

- **11.01 GPU Memory Planning**: Weight + KV cache + activation budgets; quantisation
  impact; multi-GPU placement calculator
- **11.02 Tensor Parallelism**: Column/row-parallel linear layers; Megatron-LM pattern;
  all-reduce cost model; TP vs PP comparison
- **11.03 Flash Attention**: IO complexity derivation; online softmax; tiled forward
  pass implementation; IO reduction from O(N^2) to O(N)
- **11.04 Production Observability**: Structured logging; metrics collector with
  histograms; SLO tracking with burn-rate alerting; degradation detection
- **11.05 Edge Deployment**: Device capability profiling; GGUF quantisation tiers;
  ONNX export and runtime; deployment Pareto across tiers

## 6. Exercises

1. **GGUF header parser**: Write a Python function that reads the first 1 KB of a
   `.gguf` file and extracts the model architecture fields (n_layers, hidden_size,
   vocab_size) from the key-value metadata header. Validate against the model card.

2. **ONNX quantisation**: Use `onnxruntime.quantization.quantize_dynamic` to
   produce an INT8 ONNX model from the GPT-2 export. Compare file size, forward
   pass latency, and logit accuracy against the FP32 ONNX model.

3. **Bandwidth micro-benchmark**: Implement a memory bandwidth benchmark that
   reads a 1 GB tensor from RAM and reports achieved GB/s. Compare against your
   theoretical memory bandwidth. Does your CPU achieve >80% of peak?

4. **GPU offload sweep**: Using the llama.cpp Python bindings (`llama-cpp-python`),
   run a 1B GGUF model with n_gpu_layers in [0, 8, 16, 24, 32]. Measure wall-clock
   tok/s for each setting. Does the measured speedup match the theoretical model
   in Panel 5?

5. **Device recommendation CLI**: Build a command-line tool that takes a model
   name (e.g. `llama-3-8b`) and a device name, and prints: recommended quantisation,
   estimated memory usage, estimated tok/s, and a go/no-go deployment recommendation.

---

## 7. References

- [llama.cpp project](https://github.com/ggerganov/llama.cpp)
- [GGUF format specification](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md)
- [MLC-LLM: Machine Learning Compilation for LLMs](https://mlc.ai/mlc-llm/)
- [ONNX Runtime documentation](https://onnxruntime.ai/docs/)
- [ExecuTorch (PyTorch mobile)](https://pytorch.org/executorch/)
